In [1]:
from pathlib import Path
from datetime import datetime
import io

import numpy as np
import pandas as pd
import requests

ANO = 2025
SEED = 40

PROCESSADO = Path("../data/base-processada")
PROCESSADO.mkdir(parents=True, exist_ok=True)

Path("../resultados/tabelas").mkdir(parents=True, exist_ok=True)


In [2]:
URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"


def baixar_ano(ano, tamanho_pagina=50_000):
    partes, offset = [], 0
    while True:
        params = {
            "$where": f"crash_date between '{ano}-01-01T00:00:00' and '{ano}-12-31T23:59:59'",
            "$order": "crash_date, collision_id",
            "$limit": tamanho_pagina,
            "$offset": offset,
        }
        r = requests.get(URL, params=params, timeout=120)
        r.raise_for_status()
        parte = pd.read_csv(
            io.StringIO(r.text),
            dtype={"collision_id": str, "zip_code": str},
            low_memory=False,
        )
        partes.append(parte)
        print(f"offset {offset}: {len(parte)} linhas")
        if len(parte) < tamanho_pagina:
            break
        offset += tamanho_pagina
    return pd.concat(partes, ignore_index=True)

df = baixar_ano(ANO)
print(df.shape)
print(df.columns.tolist())

offset 0: 50000 linhas
offset 50000: 35559 linhas
(85559, 29)
['crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'location', 'on_street_name', 'off_street_name', 'cross_street_name', 'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed', 'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed', 'contributing_factor_vehicle_1', 'contributing_factor_vehicle_2', 'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5', 'collision_id', 'vehicle_type_code1', 'vehicle_type_code2', 'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5']


In [3]:
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")

print(df["crash_date"].dt.month.value_counts().sort_index())
print("Fora de", ANO, ":", (df["crash_date"].dt.year != ANO).sum())
print("IDs duplicados:", df["collision_id"].duplicated().sum())
print("Datas inválidas:", df["crash_date"].isna().sum())
print(df.dtypes[["latitude", "longitude"]])

crash_date
1     6482
2     5988
3     7035
4     6924
5     7636
6     7486
7     7415
8     6992
9     7538
10    7538
11    7148
12    7377
Name: count, dtype: int64
Fora de 2025 : 0
IDs duplicados: 0
Datas inválidas: 0
latitude     float64
longitude    float64
dtype: object


In [4]:
print("Tem localização mas não tem lat/lon: ", (df["location"].notna() & df["latitude"].isna()).sum())

print("Tem lat/lon mas não tem localização: ", (df["location"].isna() & df["latitude"].notna()).sum())

Tem localização mas não tem lat/lon:  0
Tem lat/lon mas não tem localização:  0


In [5]:
df = df.drop(columns=["location"], errors="ignore")

funil = [("Linhas baixadas (2025)", len(df))]

d = df.dropna(subset=["latitude", "longitude"])
funil.append(("Com latitude e longitude", len(d)))

d = d[(d["latitude"] != 0) & (d["longitude"] != 0)]
funil.append(("Sem coordenadas zero", len(d)))

d = d[d["latitude"].between(40.49, 40.92) & d["longitude"].between(-74.27, -73.68)]
funil.append(("Dentro da caixa de NYC", len(d)))

funil.append(("Pontos únicos (lat, lon)",
              d[["latitude", "longitude"]].drop_duplicates().shape[0]))

tabela_funil = pd.DataFrame(funil, columns=["etapa", "linhas"])
tabela_funil["% do total"] = (tabela_funil["linhas"] / len(df) * 100).round(1)
print(tabela_funil.to_string(index=False))

df_limpo = d.reset_index(drop=True)
df_limpo.to_csv(PROCESSADO / f"colisoes_{ANO}_limpo.csv", index=False)
tabela_funil.to_csv("../resultados/tabelas/funil_volumetria.csv", index=False)

                   etapa  linhas  % do total
  Linhas baixadas (2025)   85559       100.0
Com latitude e longitude   84833        99.2
    Sem coordenadas zero   82918        96.9
  Dentro da caixa de NYC   82918        96.9
Pontos únicos (lat, lon)   44111        51.6


In [6]:
sem_nulo = df[df["latitude"].notna()]
print(((sem_nulo.latitude == 0) & (sem_nulo.longitude == 0)).sum(), "em (0, 0)")
print(((sem_nulo.latitude == 0) ^ (sem_nulo.longitude == 0)).sum(), "com só um dos dois zero")

1915 em (0, 0)
0 com só um dos dois zero


In [7]:
contagem = (df_limpo.groupby(["latitude", "longitude"])
                    .size().rename("n").reset_index())

print(contagem["n"].describe())
print(contagem["n"].value_counts().sort_index().head(10))
print((contagem["n"] >= 20).sum(), "pontos com 20 ou mais acidentes")
print(contagem.sort_values("n", ascending=False).head(10))

count    44111.000000
mean         1.879758
std          2.320013
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        169.000000
Name: n, dtype: float64
n
1     30024
2      6472
3      2992
4      1585
5       937
6       587
7       359
8       271
9       203
10      140
Name: count, dtype: int64
89 pontos com 20 ou mais acidentes
        latitude  longitude    n
3914   40.608757 -74.038086  169
3298   40.604153 -74.051980   55
41863  40.861866 -73.912740   50
18548  40.696033 -73.984530   42
14082  40.675735 -73.896850   42
18593  40.696198 -73.988760   39
12596  40.668507 -73.925606   35
35221  40.795220 -73.921524   34
35349  40.797370 -73.919810   33
13916  40.674854 -73.970780   33


In [8]:
df["sem_coord"] = (df["latitude"].isna() | (df["latitude"] == 0) | (df["longitude"] == 0))
print(df.groupby("borough", dropna=False)["sem_coord"].agg(["mean", "sum"]))

                   mean  sum
borough                     
BRONX          0.032056  394
BROOKLYN       0.025457  680
MANHATTAN      0.041345  599
QUEENS         0.027839  608
STATEN ISLAND  0.023977   75
NaN            0.040141  285
